In [10]:
import pandas as pd 
import sys
sys.path.insert(0,'..')
sys.path.insert(0,'../..')
from models import MetaEvaluator
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp

In [11]:
FINAL_FEATURE_FRACTION = {'powersupply': 50, 'airlines': 65, 'electricity': 15, 'rialto': 10}
elec_eval = MetaEvaluator(dataset_name="electricity", dir="henrique_st", feature_fraction=FINAL_FEATURE_FRACTION["electricity"]).fit()
powersupply_eval = MetaEvaluator(dataset_name="powersupply",dir="henrique_st", feature_fraction=FINAL_FEATURE_FRACTION["powersupply"]).fit()
airlines_eval = MetaEvaluator(dataset_name="airlines",dir="henrique_st", feature_fraction=FINAL_FEATURE_FRACTION["airlines"]).fit()
rialto_eval = MetaEvaluator(dataset_name="rialto",dir="henrique_st", feature_fraction=FINAL_FEATURE_FRACTION["rialto"]).fit()

In [ ]:
some_model = elec_eval.metrics.keys()[0]
print(some_model)
mean_results = [
    {
        "dataset": evaluator.dataset_name,
        "model": model,
        "metric": metric,
        "baseline": evaluator.results[model][f"{metric}_mse_baseline"].mean(),
        "original_mtl": evaluator.results[model][f"{metric}_original_mtl_mse"].mean(),
        "proposed_mtl": evaluator.results[model][f"{metric}_proposed_mtl_mse"].mean(),
    }
    for evaluator in (elec_eval, powersupply_eval, airlines_eval, rialto_eval)
    for model in evaluator.metrics.keys()
    for metric in ["f1-score"]
]
mean_results_df = pd.DataFrame(mean_results)
print(mean_results_df.head())
print(mean_results_df.shape)

aux = mean_results_df

TypeError: 'dict_keys' object is not subscriptable

In [ ]:
mean_results_df = mean_results_df[["baseline","original_mtl","proposed_mtl"]]
print(mean_results_df.shape)

(16, 3)


In [ ]:

transposed_results = mean_results_df.T

baseline = transposed_results.loc['baseline'].values
original_mtl = transposed_results.loc['original_mtl'].values
proposed_mtl = transposed_results.loc['proposed_mtl'].values

friedman_statistics, friedman_p_value= friedmanchisquare(baseline,original_mtl,proposed_mtl)
print(f"Friedman Statistics: {friedman_statistics}")
print(f"Friedman p-value: {friedman_p_value}")

Friedman Statistics: 18.375
Friedman p-value: 0.00010231032105679586


In [ ]:
nemenyi_results = sp.posthoc_nemenyi_friedman(mean_results_df)

Average Ranks: groups
baseline        2.8125
original_mtl    1.8750
proposed_mtl    1.3125
Name: mat, dtype: float64
Critical Difference: 0.8283755941600405


In [ ]:
print(nemenyi_results)
alpha = 0.05
print(nemenyi_results<alpha)

              baseline  original_mtl  proposed_mtl
baseline      1.000000      0.021837      0.000066
original_mtl  0.021837      1.000000      0.249493
proposed_mtl  0.000066      0.249493      1.000000
              baseline  original_mtl  proposed_mtl
baseline         False          True          True
original_mtl      True         False         False
proposed_mtl      True         False         False


In [ ]:
mean_results_df

,baseline,original_mtl,proposed_mtl
0,0.108935,0.037119,0.024179
1,0.056865,0.016308,0.011017
2,0.084411,0.026856,0.020754
3,0.086934,0.035226,0.025822
4,0.006275,0.003644,0.003655
5,0.009413,0.005532,0.004488
6,0.001423,0.000926,0.000866
7,0.003381,0.001951,0.002085
8,0.004612,0.011161,0.003921
9,0.003148,0.002711,0.002549


In [ ]:
ordered_indexes =  mean_results_df.rank(axis=1,ascending=True).astype(int)
print(ordered_indexes)

    baseline  original_mtl  proposed_mtl
0          3             2             1
1          3             2             1
2          3             2             1
3          3             2             1
4          3             1             2
5          3             2             1
6          3             2             1
7          3             1             2
8          2             3             1
9          3             2             1
10         1             3             2
11         3             1             2
12         3             2             1
13         3             1             2
14         3             2             1
15         3             2             1


In [ ]:
baseline_better = (ordered_indexes["baseline"]==1)
baseline_better_indexes = ordered_indexes[baseline_better].index
print(f"baseline_better: {baseline_better_indexes.shape[0]}")
print(aux.loc[baseline_better_indexes][["dataset","model","metric"]])

original_mtl_better = (ordered_indexes["original_mtl"]==1)
original_mtl_better_indexes = (ordered_indexes[original_mtl_better]==1).index
print(f"original_mtl_better: {original_mtl_better_indexes.shape[0]}")
print(aux.loc[original_mtl_better_indexes][["dataset","model","metric"]])

proposed_mtl_better = (ordered_indexes["proposed_mtl"]==1)
proposed_mtl_better_indexes = (ordered_indexes[proposed_mtl_better]==1).index
print(f"proposed_mtl_better: { proposed_mtl_better_indexes.shape[0]}" )

baseline_better: 1
     dataset               model metric
10  airlines  LogisticRegression  kappa
original_mtl_better: 4
        dataset                   model metric
4   powersupply  RandomForestClassifier  kappa
7   powersupply  DecisionTreeClassifier  kappa
11     airlines  DecisionTreeClassifier  kappa
13       rialto                     SVC  kappa
proposed_mtl_better: 11
